# Item 108: Verify Related Behaviours in `TestCase` Subclasses

## Notes

-   `unittest` is the built-in testing module for python
    -   Similar to the Java ecosystem’s `Junit` testing framework
-   Consider the following utility code to be tested

In [1]:
def to_str(data):
    if isinstance(data, str):
        return data
    elif isinstance(data, bytes):
        return data.decode("utf-8")
    else:
        raise TypeError(f"Must supply str or bytes, found: {data}")

-   Normally tests are then defined in a second file
    -   For demonstration we’ll be doing everything in the notebook
    -   Typically named either `test_util` or `util_test` etc.

In [2]:
%reset

from unittest import TestCase, main


# utils functionality
def to_str(data):
    if isinstance(data, str):
        return data
    elif isinstance(data, bytes):
        return data.decode("utf-8")
    else:
        raise TypeError(f"Must supply str or bytes, found: {data}")


# Testing code


class UtilsTestCase(TestCase):
    def test_to_str_bytes(self):
        self.assertEqual("hello", to_str(b"hello"))

    def test_to_str_str(self):
        self.assertEqual("hello", to_str("hello"))

    def test_failing(self):
        self.assertEqual("incorrect", to_str("hello"))


main(argv=[""], exit=False)

-   One can then run the test file via

``` shell
uv run test_util.py
```

-   For us we run the cell above directly
-   Two of the tests pass
    -   We get some error output indicating a test has failed
-   Test’s are organised via `TestCase` subclasses
    -   Tests are methods beginning with `test`
-   Any test method that runs without raising an exception is regarded
    to have passed (See [Item
    81](../../Chapter_10/Item_081/item_081.qmd))
-   Even if one test fails, all remaining tests should still run
-   To run a specific test it can be specified directly via the command
    line

``` shell
uv run test_util.py UtilsTestCase.test_to_str_bytes
```

-   Debugger can also be invoked directly in test methods for
    introspection (See [Item 114](../Item_114/item_114.qmd))
-   `TestCase` provides assertion methods for simplifying asserts
    -   `AssertEqual` for equality
    -   `AssertTrue` for truthfulness
    -   `AssertAlmostEqual` for imprecise floating point (See [Item
        113](../Item_113/item_113.qmd))
-   As always you should [read the
    docs](https://docs.python.org/3/library/unittest.html)
    -   Prefer them over a raw `assert`
    -   They will provide more contextual information for understanding
        why a test case has failed

In [3]:
%reset

from unittest import TestCase, main
# Testing code


class AssertTestCase(TestCase):
    def test_assert_helper(self):
        expected = 12
        found = 2 * 5
        self.assertEqual(expected, found)

    def test_assert_statement(self):
        expected = 12
        found = 2 * 5
        assert found == expected


main(argv=[""], exit=False)

-   If we want to verify that a method *does* raise an exception we can
    use the `assertRaises`
    -   Can also be used as a context manager (See [Item
        82](../../Chapter_10/Item_082/item_082.qmd))
    -   Gives a similar interface to a `try/except` block

In [4]:
%reset

from unittest import TestCase, main


# utils functionality
def to_str(data):
    if isinstance(data, str):
        return data
    elif isinstance(data, bytes):
        return data.decode("utf-8")
    else:
        raise TypeError(f"Must supply str or bytes, found: {data}")


# Testing code


class UtilsErrorTestCase(TestCase):
    def test_to_str_bad(self):
        with self.assertRaises(TypeError):
            to_str(object())

    def test_to_str_bad_encoding(self):
        with self.assertRaises(UnicodeDecodeError):
            to_str(b"\xfa\xfa")


main(argv=[""], exit=False)

-   Normal helper methods can be defined for complex logic
    -   Just don’t name the method starting with `test`
-   Can use the `fail` method to clarify why a test fails
    -   e.g. if an invariant is violated

In [5]:
%reset

from unittest import TestCase, main


def sum_squares(values):
    cumulative = 0
    for value in values:
        cumulative += value**2
        yield cumulative


class HelperTestCase(TestCase):
    def verify_complex_case(self, values, expected):
        expect_it = iter(expected)
        found_it = iter(sum_squares(values))
        test_it = zip(expect_it, found_it, strict=True)

        for i, (expect, found) in enumerate(test_it):
            if found != expect:
                self.fail(f"Index {i} is wrong: {found} != {expect}")

    def test_too_short(self):
        values = [1.1, 2.2]
        expected = [1.1**2]
        self.verify_complex_case(values, expected)

    def test_too_long(self):
        values = [1.1, 2.2]
        expected = [1.1**2, 1.1**2 + 2.2**2, 0]
        self.verify_complex_case(values, expected)

    def test_wrong_results(self):
        values = [1.1, 2.2, 3.3]
        expected = [
            1.1**2,
            1.1**2 + 2.2**2,
            1.1**2 + 2.2**2 + 3.3**2 + 4.4**2,
        ]
        self.verify_complex_case(values, expected)


main(argv=[""], exit=False)

-   A good pattern is *one* `TestCase` subclass for each set of related
    tests
    -   e.g. if a function has many edge cases it get’s it’s own class
-   For simple functions one test case per module may be a better
    organisation
-   Often one-to-one matching of a `TestCase` to a basic class and it’s
    methods (See [Item 109](../Item_109/item_109.qmd))
-   `subTest` helper method can be used to reduce boilerplate for
    multiple tests
    -   Helpful when writing data-driven tests
    -   Can continue to run further tests after one fails (See [Item
        110](../Item_110/item_110.qmd))
        -   similar to how different `test` methods in the same
            `TestCase` continue to run even after failure

In [6]:
%reset

from unittest import TestCase, main


# utils functionality
def to_str(data):
    if isinstance(data, str):
        return data
    elif isinstance(data, bytes):
        return data.decode("utf-8")
    else:
        raise TypeError(f"Must supply str or bytes, found: {data}")


class DataDrivenTestCase(TestCase):
    def test_good(self):
        good_cases = [
            (b"my bytes", "my bytes"),
            ("no error", b"no error"),  # this one fails
            ("other str", "other str"),
        ]

        for value, expected in good_cases:
            with self.subTest(value):
                self.assertEqual(expected, to_str(value))

    def test_bad(self):
        bad_cases = [
            (object(), TypeError),
            (b"\xfa\xfa", UnicodeDecodeError),
        ]
        for value, exception in bad_cases:
            with self.subTest(value):
                with self.assertRaises(exception):
                    to_str(value)


main(argv=[""], exit=False)

-   `unittest` is a powerful framework
    -   However it does it have it’s limits
    -   E.g. it’s java-style focus on OOP and naming conventions
-   When you outgrow `unittest` consider
    [`pytest`](https://docs.pytest.org/en/stable/)
    -   Community-developed test framework
        -   Very popular
    -   Uses a more functional style
    -   Has a large number of extensions and plugins to provide more
        testing power

## Things to Remember

-   Tests can be created using the `unittest` built-in framework
    -   Subclass the `TestCase` class
    -   Define one method per behaviour being tested
        -   It’s name must start with `test`
-   Use helper methods defined by `TestCase` such as `assertEqual` to
    confirm expected behaviours and get meaningful output when tests
    fail
    -   Prefer them over the built-in `assert` function
-   Use the `SubTest` helper method to write data-driven tests with
    reduced boilerplate